In [1]:
import bolero
import torch
from bolero.tl.generic.module_lora_cond import *
from einops import einsum, rearrange, repeat
from torchinfo import summary

# bolero.init(num_cpus=32)

## Linear

In [6]:
linear = torch.nn.Linear(10, 100)
lora_linear = ConditionalLoRALinear.from_nn(
    linear,
    emb_input_features=30,
    hidden_dim=64,
    rank=2,
    hidden_layers=1,
    output_layer_groups=1,
)

x = torch.ones((64, 10))
emb = torch.ones((64, 30))
summary(
    lora_linear,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    row_settings=["var_names"],
)

Layer (type (var_name))                            Input Shape               Output Shape              Param #
ConditionalLoRALinear (ConditionalLoRALinear)      --                        [64, 100]                 1,100
├─EmbeddingMLP (lora_A_module)                     --                        [64, 2, 10]               1
│    └─Sequential (mlp)                            [64, 30]                  [64, 20]                  --
│    │    └─Linear (0)                             [64, 30]                  [64, 64]                  1,984
│    │    └─BatchNorm1d (1)                        [64, 64]                  [64, 64]                  128
│    │    └─GELU (2)                               [64, 64]                  [64, 64]                  --
│    │    └─Linear (3)                             [64, 64]                  [64, 64]                  4,160
│    │    └─BatchNorm1d (4)                        [64, 64]                  [64, 64]                  128
│    │    └─GELU (5)           

## Conv

In [3]:
conv = nn.Conv1d(in_channels=10, out_channels=20, kernel_size=15, groups=2, padding=7)
lora_conv = ConditionalLoRAConv.from_nn(
    conv,
    emb_input_features=30,
    hidden_dim=64,
    rank=2,
    hidden_layers=1,
    output_layer_groups=2,
)
x = torch.ones((64, 10, 100))
emb = torch.ones((64, 30))
summary(
    lora_conv,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    row_settings=["var_names"],
)

Layer (type (var_name))                       Input Shape               Output Shape              Param #
ConditionalLoRAConv (ConditionalLoRAConv)     --                        [64, 20, 100]             1,520
├─EmbeddingMLP (lora_A_module)                --                        [64, 30, 75]              1
│    └─Sequential (mlp)                       [64, 30]                  [64, 2250]                --
│    │    └─Linear (0)                        [64, 30]                  [64, 64]                  1,984
│    │    └─BatchNorm1d (1)                   [64, 64]                  [64, 64]                  128
│    │    └─GELU (2)                          [64, 64]                  [64, 64]                  --
│    │    └─Linear (3)                        [64, 64]                  [64, 64]                  4,160
│    │    └─BatchNorm1d (4)                   [64, 64]                  [64, 64]                  128
│    │    └─GELU (5)                          [64, 64]                  [64,

In [4]:
conv = nn.Conv2d(in_channels=10, out_channels=20, kernel_size=15, groups=2, padding=7)
lora_conv = ConditionalLoRAConv.from_nn(
    conv,
    emb_input_features=30,
    hidden_dim=256,
    rank=16,
    hidden_layers=0,
    output_layer_groups=1,
)
x = torch.ones((64, 10, 100, 100))
emb = torch.ones((64, 30))
summary(
    lora_conv,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    row_settings=["var_names"],
)

Layer (type (var_name))                       Input Shape               Output Shape              Param #
ConditionalLoRAConv (ConditionalLoRAConv)     --                        [64, 20, 100, 100]        22,520
├─EmbeddingMLP (lora_A_module)                --                        [64, 30, 75]              1
│    └─Sequential (mlp)                       [64, 30]                  [64, 2250]                --
│    │    └─Linear (0)                        [64, 30]                  [64, 64]                  1,984
│    │    └─BatchNorm1d (1)                   [64, 64]                  [64, 64]                  128
│    │    └─GELU (2)                          [64, 64]                  [64, 64]                  --
│    │    └─Linear (3)                        [64, 64]                  [64, 64]                  4,160
│    │    └─BatchNorm1d (4)                   [64, 64]                  [64, 64]                  128
│    │    └─GELU (5)                          [64, 64]                  [64

## COrigami

In [5]:
model = torch.load("model.pt", weights_only=False)

### Encoder

In [6]:
summary(
    model.encoder,
    input_size=[16, 7, 2097152],
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=4,
    row_settings=["var_names"],
)

Layer (type (var_name))                  Input Shape               Output Shape              Param #
EncoderSplit (EncoderSplit)              [16, 7, 2097152]          [16, 256, 256]            2,930,400
├─Sequential (conv_start_seq)            --                        --                        --
│    └─Conv1d (0)                        [16, 5, 2097152]          [16, 16, 1048576]         256
│    └─BatchNorm1d (1)                   [16, 16, 1048576]         [16, 16, 1048576]         32
│    └─ReLU (2)                          [16, 16, 1048576]         [16, 16, 1048576]         --
├─Sequential (conv_start_epi)            --                        --                        --
│    └─Conv1d (0)                        [16, 2, 2097152]          [16, 16, 1048576]         112
│    └─BatchNorm1d (1)                   [16, 16, 1048576]         [16, 16, 1048576]         32
│    └─ReLU (2)                          [16, 16, 1048576]         [16, 16, 1048576]         --
├─Sequential (res_blocks_s

In [7]:
model.encoder = convert_to_conditional_lora_model(
    model.encoder,
    emb_input_features=30,
    hidden_dim=32,
    hidden_layers=2,
    output_layer_groups=4,
    convert_linear=True,
    convert_conv=True,
    rank=16,
    verbose=True
)
model.encoder = model.encoder.to('cuda')

Converting 'conv_start.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.res.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.res.3' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.res.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.res.3' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.res.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.res.3' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.res.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.res.3' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.4.scale.0' <Conv1d> to <Co

In [8]:
x = torch.ones(16, 7, 2097152).to('cuda')
emb = torch.ones(16, 30).to('cuda')

summary(
    model.encoder,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=4,
    row_settings=["var_names"],
)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #
EncoderSplit (EncoderSplit)                             --                        [16, 256, 256]            18,975,786
├─Sequential (conv_start_seq)                           --                        --                        --
│    └─ConditionalLoRAConv (0)                          [16, 5, 2097152]          [16, 16, 1048576]         256
│    │    └─EmbeddingMLP (lora_A_module)                --                        [16, 48, 15]              1
│    │    │    └─Sequential (mlp)                       [16, 30]                  [16, 720]                 9,776
│    │    └─EmbeddingMLP (lora_B_module)                --                        [16, 16, 48]              1
│    │    │    └─Sequential (mlp)                       [16, 30]                  [16, 768]                 10,208
│    └─BatchNorm1d (1)                                  [16, 16, 1048576]         [16, 16, 10

### Attn

In [9]:
summary(
    model.attn,
    input_size=[16, 256, 256],
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=4,
    row_settings=["var_names"],
)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #
AttnModule (AttnModule)                                 [16, 256, 256]            [16, 256, 256]            --
├─PositionalEncoding (pos_encoder)                      [16, 256, 256]            [16, 256, 256]            --
│    └─Dropout (dropout)                                [16, 256, 256]            [16, 256, 256]            --
├─TransformerEncoder (module)                           [16, 256, 256]            [16, 256, 256]            --
│    └─ModuleList (layers)                              --                        --                        --
│    │    └─TransformerLayer (0)                        [16, 256, 256]            [16, 256, 256]            --
│    │    │    └─LayerNorm (norm1)                      [16, 256, 256]            [16, 256, 256]            512
│    │    │    └─MultiheadAttention (self_attn)         [16, 256, 256]            [16, 256, 256]          

In [10]:
model.attn = convert_to_conditional_lora_model(
    model.attn,
    emb_input_features=30,
    hidden_dim=32,
    hidden_layers=2,
    output_layer_groups=4,
    convert_linear=True,
    convert_conv=False,
    rank=16,
    # exclude_name_patterns=['.+self_attn.out_proj$'],
    verbose=True
)
model.attn = model.attn.to('cuda')

Converting 'module.layers.0.self_attn.out_proj' <NonDynamicallyQuantizableLinear> to <ConditionalLoRALinear> module
Converting 'module.layers.0.linear1' <Linear> to <ConditionalLoRALinear> module
Converting 'module.layers.0.linear2' <Linear> to <ConditionalLoRALinear> module
Converting 'module.layers.1.self_attn.out_proj' <NonDynamicallyQuantizableLinear> to <ConditionalLoRALinear> module
Converting 'module.layers.1.linear1' <Linear> to <ConditionalLoRALinear> module
Converting 'module.layers.1.linear2' <Linear> to <ConditionalLoRALinear> module
Converting 'module.layers.2.self_attn.out_proj' <NonDynamicallyQuantizableLinear> to <ConditionalLoRALinear> module
Converting 'module.layers.2.linear1' <Linear> to <ConditionalLoRALinear> module
Converting 'module.layers.2.linear2' <Linear> to <ConditionalLoRALinear> module
Converting 'module.layers.3.self_attn.out_proj' <NonDynamicallyQuantizableLinear> to <ConditionalLoRALinear> module
Converting 'module.layers.3.linear1' <Linear> to <Condit

In [11]:
x = torch.ones(16, 256, 256).to('cuda')
emb = torch.ones(16, 30).to('cuda')

summary(
    model.attn,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=4,
)

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
AttnModule                                                   --                        [16, 256, 256]            --
├─PositionalEncoding: 1-1                                    [16, 256, 256]            [16, 256, 256]            --
│    └─Dropout: 2-1                                          [16, 256, 256]            [16, 256, 256]            --
├─TransformerEncoder: 1-2                                    [16, 256, 256]            [16, 256, 256]            --
│    └─ModuleList: 2-2                                       --                        --                        --
│    │    └─TransformerLayer: 3-1                            [16, 256, 256]            [16, 256, 256]            --
│    │    │    └─LayerNorm: 4-1                              [16, 256, 256]            [16, 256, 256]            (512)
│    │    │    └─MultiheadAttention: 4-2                     [16

### Decoder

In [12]:
summary(
    model.decoder,
    input_size=[16, 512, 256, 256],
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=3,
    row_settings=["var_names"],
)

Layer (type (var_name))                  Input Shape               Output Shape              Param #
Decoder (Decoder)                        [16, 512, 256, 256]       [16, 1, 256, 256]         --
├─Sequential (conv_start)                --                        --                        --
│    └─Conv2d (0)                        [16, 512, 256, 256]       [16, 256, 256, 256]       1,179,904
│    └─BatchNorm2d (1)                   [16, 256, 256, 256]       [16, 256, 256, 256]       512
│    └─ReLU (2)                          [16, 256, 256, 256]       [16, 256, 256, 256]       --
├─Sequential (res_blocks)                --                        --                        --
│    └─ResBlockDilated (0)               [16, 256, 256, 256]       [16, 256, 256, 256]       --
│    │    └─Sequential (res)             --                        --                        1,181,184
│    │    └─ReLU (relu)                  [16, 256, 256, 256]       [16, 256, 256, 256]       --
│    └─ResBlockDilat

In [13]:
model.decoder = convert_to_conditional_lora_model(
    model.decoder,
    emb_input_features=30,
    hidden_dim=32,
    hidden_layers=2,
    output_layer_groups=4,
    convert_linear=False,
    convert_conv=True,
    rank=16,
    verbose=True
)
model.decoder = model.decoder.to('cuda')

Converting 'conv_start.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.res.3' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.res.3' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.res.3' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.res.3' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.4.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.4.res.3' <Conv2d> to <ConditionalLoRAConv> module
Converting 'conv_end' <Conv2d> to <ConditionalLoRAConv> module


In [14]:
x = torch.ones(16, 512, 256, 256).to('cuda')
emb = torch.ones(16, 30).to('cuda')

summary(
    model.decoder,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=4,
    row_settings=["var_names"],
)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #
Decoder (Decoder)                                       --                        [16, 1, 256, 256]         --
├─Sequential (conv_start)                               --                        --                        --
│    └─ConditionalLoRAConv (0)                          [16, 512, 256, 256]       [16, 256, 256, 256]       1,179,904
│    │    └─EmbeddingMLP (lora_A_module)                --                        [16, 48, 1536]            1
│    │    │    └─Sequential (mlp)                       [16, 30]                  [16, 73728]               666,848
│    │    └─EmbeddingMLP (lora_B_module)                --                        [16, 768, 48]             1
│    │    │    └─Sequential (mlp)                       [16, 30]                  [16, 36864]               335,072
│    └─BatchNorm2d (1)                                  [16, 256, 256, 256]       [16, 256, 

## Partially Conditional LoRA

These are just examples for how to use the function, I'm not sure how well it will work. But the idea is that not all layer needs to be conditional, a mixture of normal and conditional LoRA could perform equivalently than fully conditional while reduce the parameter numbers.

In [6]:
model = torch.load("model.pt", weights_only=False)

In [7]:
# only convert the scale Conv2d in res_block to CondLoRA, other Conv2d to normal LoRA
model.encoder = convert_to_conditional_lora_model(
    model.encoder,
    emb_input_features=30,
    hidden_dim=32,
    hidden_layers=2,
    output_layer_groups=8,
    convert_linear=True,
    convert_conv=True,
    rank=4,
    verbose=True,
    default_conditional=False,
    include_cond_lora_patterns=[".+.scale.0"]
)
model.encoder = model.encoder.to('cuda')

Converting 'conv_start.0' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.0.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.res.0' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.0.res.3' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.1.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.res.0' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.1.res.3' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.2.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.res.0' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.2.res.3' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.3.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.res.0' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.3.res.3' <Conv1d> to <LoRAConv> module
Converting 'res_blocks.4.scale.0' <Conv1d> to <ConditionalLoRAConv> module
Converting 'res_blocks.4.res.0' <Conv1d> to <LoRAConv> module
Converting 

In [8]:
x = torch.ones(16, 7, 2097152).to('cuda')
emb = torch.ones(16, 30).to('cuda')

summary(
    model.encoder,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=6,
    row_settings=["var_names"],
)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #
EncoderSplit (EncoderSplit)                             --                        [16, 256, 256]            3,973,256
├─Sequential (conv_start_seq)                           --                        --                        --
│    └─LoRAConv (0)                                     [16, 5, 2097152]          [16, 16, 1048576]         372
│    │    └─Conv1d (conv)                               [16, 5, 2097152]          [16, 16, 1048576]         (256)
│    └─BatchNorm1d (1)                                  [16, 16, 1048576]         [16, 16, 1048576]         (32)
│    └─ReLU (2)                                         [16, 16, 1048576]         [16, 16, 1048576]         --
├─Sequential (conv_start_epi)                           --                        --                        --
│    └─LoRAConv (0)                                     [16, 2, 2097152]          [16, 16, 104

In [39]:
# only convert the first Conv2d in res_block to CondLoRA, other Conv2d to normal LoRA
model.decoder = convert_to_conditional_lora_model(
    model.decoder,
    emb_input_features=30,
    hidden_dim=256,
    hidden_layers=2,
    output_layer_groups=8,
    convert_linear=False,
    convert_conv=True,
    rank=12,
    verbose=True, 
    default_conditional=False,
    include_cond_lora_patterns=['res_blocks.\d+.res.0']
)
model.decoder = model.decoder.to('cuda')

Converting 'conv_start.0' <Conv2d> to <LoRAConv> module
Converting 'res_blocks.0.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.0.res.3' <Conv2d> to <LoRAConv> module
Converting 'res_blocks.1.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.1.res.3' <Conv2d> to <LoRAConv> module
Converting 'res_blocks.2.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.2.res.3' <Conv2d> to <LoRAConv> module
Converting 'res_blocks.3.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.3.res.3' <Conv2d> to <LoRAConv> module
Converting 'res_blocks.4.res.0' <Conv2d> to <ConditionalLoRAConv> module
Converting 'res_blocks.4.res.3' <Conv2d> to <LoRAConv> module
Converting 'conv_end' <Conv2d> to <LoRAConv> module


In [40]:
x = torch.ones(16, 512, 256, 256).to('cuda')
emb = torch.ones(16, 30).to('cuda')

summary(
    model.decoder,
    input_data={"x": x, "embedding": emb},
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=4,
    row_settings=["var_names"],
)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #
Decoder (Decoder)                                       --                        [16, 1, 256, 256]         --
├─Sequential (conv_start)                               --                        --                        --
│    └─LoRAConv (0)                                     [16, 512, 256, 256]       [16, 256, 256, 256]       82,944
│    │    └─Conv2d (conv)                               [16, 512, 256, 256]       [16, 256, 256, 256]       (1,179,904)
│    └─BatchNorm2d (1)                                  [16, 256, 256, 256]       [16, 256, 256, 256]       (512)
│    └─ReLU (2)                                         [16, 256, 256, 256]       [16, 256, 256, 256]       --
├─Sequential (res_blocks)                               --                        --                        --
│    └─ResBlockDilated (0)                              [16, 256, 256, 256]       [16, 256,

In [31]:
model.decoder.res_blocks[0].res[0].lora_A_module.output_shape

(48, 768)

In [32]:
model.decoder.res_blocks[0].res[0].lora_B_module.output_shape

(768, 48)